# PART 7-2. 리뷰 감성 분석 ② 딥러닝 (Embedding · LSTM)

> **핵심 질문: 딥러닝은 머신러닝을 이길까?**
##### NLP = Natural Language Processing (자연어 처리)
## 📋 한눈에 보기

**작업 흐름**
```
1. 7-1과 같은 리뷰 · 같은 test 준비
2. train에서 validation 떼어 두기
3. Tokenizer → Sequence → Padding
4. 딥러닝 ① Embedding 평균 (EarlyStopping)
5. 딥러닝 ② Bi-LSTM (단어 순서 반영)
6. 7-1 머신러닝과 비교
7. 배포용 저장
```

| 항목 | 내용 |
|---|---|
| 사용 데이터 | 7-1과 같은 리뷰 37,139건 |
| 결과물 | `part7_2_sentiment_model.keras` (약 2.5MB) + `part7_2_tokenizer.pkl` (비교용) |

**🎯 핵심 (꼭 가져갈 것)**
- 딥러닝 점수도 **단순한 기준선**(전부 긍정, 7-1 머신러닝)과 비교해야 의미가 있습니다.
- test·validation은 처음에 떼어 두고 절대 손대지 않습니다.

**💡 인사이트 (이 파트에서 확인한 사실)**
- 머신러닝 0.917, Embedding 평균 0.924, Bi-LSTM 0.919 → **거의 같습니다.**
- Olist 리뷰는 짧고 핵심 단어로 판단되는 문제라, 단어 순서를 읽어도 얻을 것이 적습니다.
- 옵티마이저를 빼고 저장하면 파일이 약 7.5MB → 2.5MB로 줄어듭니다.

---

## 수업 목표
- 리뷰 텍스트를 숫자로 변환하고 딥러닝으로 긍정/부정을 예측합니다.
- Tokenizer, Sequence, Padding, Embedding의 개념을 이해합니다.
- PART 7-1(머신러닝)과 **같은 리뷰 · 같은 test**로 결과를 비교합니다.

## 수업 진행 포인트
| 구분 | 설명 |
|---|---|
| 데이터 관점 | 문장(텍스트)을 AI가 이해할 수 있는 숫자로 바꾸는 과정입니다. |
| 코드 관점 | Tokenizer → pad_sequences → Sequential 모델 순서를 익힙니다. |
| AI 관점 | 딥러닝은 숫자만 처리할 수 있으므로 문장을 반드시 변환해야 합니다. |
---


## 📌 이 파트에서 사용하는 API 흐름 (Keras)

| 단계 | 역할 |
|---|---|
| ① Define | 모델 정의 (레이어 쌓기) |
| ② Compile | 손실함수 · 옵티마이저 설정 |
| ③ Fit | 학습 |
| ④ Evaluate | 평가 |
| ⑤ Predict | 예측 |


## 핵심 개념

| 개념 | 설명 |
|---|---|
| Tokenizer | 문장을 단어 단위로 나누고 각 단어에 번호를 붙임 |
| Sequence | 문장을 번호 배열로 변환 |
| Padding | 모든 문장 길이를 동일하게 맞춤 (짧으면 0 채우기) |
| Embedding | 번호를 의미 있는 벡터로 변환 |
| OOV | 단어집에 없는 단어 처리 토큰 |
| EarlyStopping | 검증 성능이 더 나아지지 않으면 학습을 멈추고, 가장 좋았던 시점으로 되돌림 |
| LSTM | 문장을 앞에서부터 **순서대로** 읽으며 앞 내용을 기억하는 층 |

In [ ]:
# 셀 1. Google Drive 연결

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 셀 2. 라이브러리 불러오기

import os
import pickle
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, GlobalAveragePooling1D, Dense, Dropout,
    Bidirectional, LSTM,          # 단어 순서를 읽는 층 (셀 13-1에서 사용)
)
from tensorflow.keras.callbacks import EarlyStopping   # 검증 성능이 멈추면 학습 중단

# 딥러닝은 시작 가중치가 랜덤이라 실행할 때마다 결과가 조금씩 달라집니다.
# set_random_seed(42) : 랜덤 값을 고정해서 결과를 최대한 비슷하게 재현합니다.
# (GPU·라이브러리 버전에 따라 소수점 둘째 자리 정도는 달라질 수 있습니다.)
import tensorflow as tf
tf.keras.utils.set_random_seed(42)

base_path = "/content/drive/MyDrive/Colab Notebooks/Data_Analysis/Olist"
model_path = f"{base_path}/model"
os.makedirs(model_path, exist_ok=True)

In [ ]:
# 셀 3. 데이터 불러오기
df = pd.read_csv(f"{base_path}/data/olist_master_data.csv")

# 감성분석에 필요한 컬럼만 선택
# 통합 데이터는 주문 1건이 여러 행이라 같은 리뷰가 여러 번 들어 있습니다.
# review_id 기준으로 중복을 지워야 같은 리뷰가 train과 test에 동시에 들어가지 않습니다.
review_df = df.drop_duplicates("review_id")[["review_comment_message", "review_score"]].copy()
print(f"전체 리뷰 수: {len(review_df):,}")

전체 리뷰 수: 98,411


In [ ]:
# 셀 4. 데이터 전처리

# 1. 리뷰 내용이 없는 행 제거
review_df = review_df.dropna()
print(f"결측치 제거 후: {len(review_df):,}")

# 2. 중립(3점) 제거 → 긍정/부정 이분법 학습을 위해
review_df = review_df[review_df["review_score"] != 3]
print(f"3점 제거 후: {len(review_df):,}")

# 3. 긍정/부정 라벨 생성
#    4~5점 → 긍정(1),  1~2점 → 부정(0)
review_df["label"] = review_df["review_score"].apply(
    lambda score: 1 if score >= 4 else 0
)

print("\n라벨 분포:")
print(review_df["label"].value_counts())
# ▶ 긍정이 부정보다 약 2.5배 많습니다 → 불균형 처리 필요

결측치 제거 후: 40,668
3점 제거 후: 37,139

라벨 분포:
label
1    26380
0    10759
Name: count, dtype: int64


In [ ]:
# 셀 5. train / test 분리
# 중요: test 데이터는 시험지 역할 → 절대 수정하지 않습니다.

X = review_df["review_comment_message"]
y = review_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # 긍정/부정 비율을 train/test에 동일하게 유지
)

print(f"train 크기: {len(X_train):,}")
print(f"test 크기:  {len(X_test):,}")

train 크기: 29,711
test 크기:  7,428


In [ ]:
# 셀 6. validation 데이터 떼어 두기 + 불균형 보정 가중치(class_weight) 계산
# test와 validation은 건드리지 않습니다!

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,       # train의 20%를 검증용(validation)으로
    random_state=42,
    stratify=y_train
)
print(f"학습용(train)     : {len(X_train):,}")
print(f"검증용(validation): {len(X_val):,}")
print(f"학습용 라벨 분포  : {y_train.value_counts().to_dict()}")

# class_weight : 수가 적은 부정(0) 리뷰를 틀리면 더 큰 벌점을 줍니다.
# 공식 : 전체 개수 / (클래스 수 × 그 클래스 개수)  → 적은 클래스일수록 가중치가 커짐
n_total = len(y_train)
class_weight = {
    0: n_total / (2 * (y_train == 0).sum()),
    1: n_total / (2 * (y_train == 1).sum()),
}
print(f"class_weight      : {{0: {class_weight[0]:.2f}, 1: {class_weight[1]:.2f}}}")
# ▶ 부정 리뷰 1개를 틀리면 긍정 리뷰 약 2.5개를 틀린 것만큼 벌점을 받습니다.
#   데이터를 복제(resample)하지 않아도 불균형을 보정할 수 있습니다.

학습용(train)     : 23,768
검증용(validation): 5,943
학습용 라벨 분포  : {1: 16883, 0: 6885}
class_weight      : {0: 1.73, 1: 0.70}


In [ ]:
# 셀 7. Tokenizer - 단어집 만들기
# 각 단어에 번호를 붙이는 과정입니다.
# 예) "I love it" → {"I":23, "love":5, "it":17}

MAX_WORDS = 10000  # 자주 나오는 단어 10,000개만 사용
MAX_LEN   = 50     # 문장 길이를 50개 단어로 고정 (Olist 리뷰는 대부분 8단어 안팎, 가장 긴 리뷰도 50단어 이하)

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"  # 단어집에 없는 단어를 <OOV>로 처리
)

# train 데이터로만 단어집 학습 (test는 절대 사용 금지)
tokenizer.fit_on_texts(X_train)

word_index = tokenizer.word_index
print(f"학습된 단어 수: {len(word_index):,}")
print("자주 나오는 단어 TOP 10:")
for word, idx in list(word_index.items())[:10]:
    print(f"  {word}: {idx}")

# 문장 길이(단어 수) 분포 확인 → MAX_LEN을 정하는 근거
lengths = [len(s) for s in tokenizer.texts_to_sequences(X_train)]
print()
print(f"문장 길이(단어 수)  중앙값 {np.median(lengths):.0f}  /  95% 지점 {np.percentile(lengths, 95):.0f}  /  최대 {max(lengths)}")
# ▶ 최대 길이보다 MAX_LEN이 크면 잘리는 문장이 없습니다. 너무 크면 0(패딩)만 늘어나 계산이 낭비됩니다.

학습된 단어 수: 11,784
자주 나오는 단어 TOP 10:
  <OOV>: 1
  produto: 2
  o: 3
  e: 4
  a: 5
  do: 6
  de: 7
  não: 8
  prazo: 9
  muito: 10

문장 길이(단어 수)  중앙값 8  /  95% 지점 33  /  최대 44


```
이 데이터의 학습 문장에서 찾은 단어 : 11,784개 (위 '학습된 단어 수' 출력)
        ↓
자주 나오는 단어만 선택 (num_words=10,000)
        ↓
번호가 10,000보다 작은 단어만 사용 (1번 <OOV> 포함 → 실제 단어는 약 1만 개)
너무 드물게 나오는 단어까지 전부 사용하면 계산량이 커지고 학습도 복잡해지기 때문

나머지 드문 단어(약 1,785개)와 단어집에 없는 단어는 oov_token="<OOV>"(1번)로 처리
```

In [ ]:
# 셀 8. 문장 → 숫자 배열(Sequence)로 변환

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq   = tokenizer.texts_to_sequences(X_val)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

# 변환 예시 확인
print("원본 문장:", X_train.iloc[0])
print("숫자 변환:", X_train_seq[0])

원본 문장: Não veio com tudo que disse que vinha , nas informações do produto no vendedor , informa que vem com gelo térmico , mas veio sem
숫자 변환: [8, 29, 15, 35, 11, 972, 11, 1922, 217, 475, 6, 2, 17, 128, 1000, 11, 313, 15, 3103, 3104, 33, 29, 65]


In [ ]:
# 셀 9. Padding - 문장 길이 통일
# 딥러닝은 입력 길이가 모두 같아야 합니다.
# 짧은 문장은 뒤에 0을 채우고(padding), 긴 문장은 뒤를 잘라냅니다(truncating).

X_train_pad = pad_sequences(
    X_train_seq, maxlen=MAX_LEN,
    padding="post",    # 뒤에 0 채우기
    truncating="post"  # 뒤에서 자르기
)

X_val_pad = pad_sequences(
    X_val_seq, maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq, maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

print(f"train 패딩 결과: {X_train_pad.shape}")
# ▶ (학습 데이터 수, 50) → 모든 문장이 50개 숫자로 통일됩니다.

train 패딩 결과: (23768, 50)


```
패딩을 쓰는 이유
딥러닝 모델에 들어가는 입력 길이를 모두 같게 맞추기 위해서

문장1 → 단어 8개
문장2 → 단어 30개
문장3 → 단어 60개

길이가 제각각이면 한 번에 묶어서 학습하기 어렵습니다. 그래서 MAX_LEN = 50으로 정하면:
8개 문장  → 뒤에 0을 채워서 50개
30개 문장 → 뒤에 0을 채워서 50개
60개 문장 → 뒤 10개를 잘라서 50개

MAX_LEN은 위 셀의 '문장 길이 분포'를 보고 정합니다.
너무 짧으면 → 긴 리뷰의 뒷부분이 잘려 정보가 사라짐
너무 길면   → 대부분이 0(패딩)이라 계산만 늘어남

학습 문장이 23,768개 있고, 각 문장을 숫자 50개 길이로 통일했다.
```


### 🔵 ① Define → 모델 정의


In [ ]:
# 셀 10. 딥러닝 모델 구성 ① Embedding 평균 모델

model = Sequential([
    # Embedding: 단어 번호를 64차원 벡터로 변환 (단어의 의미를 표현)
    Embedding(input_dim=MAX_WORDS, output_dim=64),
    # ※ 예전 코드의 input_length=... 는 최신 Keras에서 쓰지 않습니다. (입력 길이는 학습할 때 자동으로 정해짐)

    # GlobalAveragePooling1D: 50개 단어 벡터의 '평균'을 내서 하나로 요약
    # ⚠️ 평균을 내면 단어 순서 정보가 사라집니다. → PART 7-1 TF-IDF처럼 '어떤 단어가 있는가'만 봅니다.
    GlobalAveragePooling1D(),

    # Dropout: 학습 중 랜덤하게 일부 뉴런을 끔 → 과적합 방지
    Dropout(0.3),

    # Dense: 완전연결층 (relu = 음수는 0으로 만드는 함수)
    Dense(64, activation="relu"),
    Dropout(0.3),

    # 출력층: sigmoid = 0~1 확률값 출력 (0.5 이상이면 긍정)
    Dense(1, activation="sigmoid"),
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

? 와 (unbuilt) 는 아직 입력 데이터 shape가 확정되지 않아 모델이 완전히 생성(build)되지 않았다는 의미이며, 학습하면 자동으로 해결됩니다.


### 🔵 ② Compile → 모델 컴파일


In [ ]:
# 셀 11. 모델 컴파일
# compile()에서 모델이 어떻게 학습하고, 어떻게 틀림을 계산하고, 무엇으로 성능을 볼지 정해주는 것
# 학습 알고리즘과 평가 기준을 모델에 지정한다
model.compile(
    optimizer="adam",           # 학습률 자동 조정 옵티마이저
    loss="binary_crossentropy", # 이진 분류용 손실 함수
    metrics=["accuracy"]        # 평가 지표: 정확도
)


### 🔵 ③ Fit → 학습


In [ ]:
# 셀 11-1. 학습 데이터 모양 확인

print(X_train_pad.shape)  # (23768, 50) → 학습용 문장 23,768개 × 길이 50
print(y_train.shape)      # (23768,)    → 정답 23,768개
print(X_val_pad.shape)    # (5943, 50)  → 검증용 문장 5,943개

(23768, 50)
(23768,)
(5943, 50)


```
epoch = 학습 데이터 전체를 한 번 다 보는 것

학습 데이터      : 23,768개 (긍정 16,883 + 부정 6,885, 부정은 class_weight로 약 2.5배 중요하게)
검증 데이터      : 5,943개  (학습에 사용 안 함)

23,768개 문제집을 여러 번 공부하고, 한 번 볼 때마다 따로 떼어 둔 5,943개로 중간시험을 본다
```

### 몇 번 반복해야 할까? → EarlyStopping

epoch를 무조건 많이 하면 문제집을 **외워 버려서**(과적합) 중간시험 점수가 오히려 떨어집니다.
반대로 너무 적으면 덜 배웁니다. 그리고 중간시험 점수는 epoch마다 **오르락내리락**합니다.

```
(예시 — 실제 숫자는 아래 셀 12의 실행 결과를 보세요)
epoch 1 → val_loss 0.34
epoch 2 → val_loss 0.28   ← 가장 좋음
epoch 3 → val_loss 0.31   (나빠짐 1번)
epoch 4 → val_loss 0.32   (나빠짐 2번) → 멈춤! 그리고 epoch 2의 가중치로 되돌림
```

| 설정 | 의미 |
|---|---|
| `monitor="val_loss"` | 검증 데이터의 손실(틀린 정도)을 지켜봅니다. |
| `patience=2` | 2번 연속 좋아지지 않으면 멈춥니다. |
| `restore_best_weights=True` | 멈춘 뒤 **가장 좋았던 epoch의 가중치**로 되돌립니다. |

> 📌 EarlyStopping이 없으면 마지막 epoch가 우연히 좋았는지 나빴는지에 따라 test 점수가 크게 달라집니다.
> 그래서 epochs는 넉넉히(10) 주고, 언제 멈출지는 검증 데이터에 맡깁니다.

In [ ]:
# 셀 12. 모델 학습 (약 30초~1분)

early_stop = EarlyStopping(
    monitor="val_loss",           # 검증 손실을 지켜봄
    patience=2,                   # 2번 연속 좋아지지 않으면 멈춤
    restore_best_weights=True,    # 가장 좋았던 시점의 가중치로 되돌림
)

history = model.fit(
    X_train_pad, y_train,
    epochs=10,            # 최대 10번 반복 (보통 그 전에 EarlyStopping이 멈춤)
    batch_size=64,        # 한 번에 64개씩 처리, 64문장마다 한 번씩 파라미터를 업데이트
    validation_data=(X_val_pad, y_val),  # 미리 떼어 둔 검증 데이터로 매 epoch 성능 확인
    class_weight=class_weight,           # 부정 리뷰를 더 중요하게 학습 (불균형 보정)
    callbacks=[early_stop],              # 학습 도중 멈출 조건
    verbose=2             # epoch마다 결과를 한 줄씩 출력
)

best_epoch = int(np.argmin(history.history["val_loss"])) + 1
print(f"\n실제 학습한 epoch 수 : {len(history.history['loss'])}")
print(f"가장 좋았던 epoch    : {best_epoch}  (이 시점의 가중치를 사용합니다)")
# ▶ val_accuracy가 accuracy와 비슷하게 올라가면 정상 학습입니다.
# ▶ accuracy(학습)는 계속 오르는데 val_loss가 다시 커지면 과적합 신호입니다. → EarlyStopping이 그 지점에서 멈춥니다.

Epoch 1/10
372/372 - 8s - 23ms/step - accuracy: 0.8261 - loss: 0.4580 - val_accuracy: 0.9038 - val_loss: 0.2771
Epoch 2/10
372/372 - 2s - 5ms/step - accuracy: 0.9024 - loss: 0.2617 - val_accuracy: 0.9191 - val_loss: 0.2236
Epoch 3/10
372/372 - 1s - 4ms/step - accuracy: 0.9202 - loss: 0.2172 - val_accuracy: 0.9253 - val_loss: 0.2015
Epoch 4/10
372/372 - 1s - 3ms/step - accuracy: 0.9278 - loss: 0.1957 - val_accuracy: 0.9282 - val_loss: 0.1911
Epoch 5/10
372/372 - 1s - 3ms/step - accuracy: 0.9378 - loss: 0.1779 - val_accuracy: 0.9293 - val_loss: 0.1901
Epoch 6/10
372/372 - 1s - 3ms/step - accuracy: 0.9443 - loss: 0.1639 - val_accuracy: 0.9266 - val_loss: 0.1921
Epoch 7/10
372/372 - 1s - 3ms/step - accuracy: 0.9486 - loss: 0.1532 - val_accuracy: 0.9295 - val_loss: 0.1924

실제 학습한 epoch 수 : 7
가장 좋았던 epoch    : 5  (이 시점의 가중치를 사용합니다)


```
딥러닝 학습은
처음 학습 시작할 때 가중치 같은 파라미터는 보통 작은 랜덤값으로 초기화된다.
파라미터(가중치)를 한 번 계산해서 끝내는 게 아니라, 조금씩 수정해가며 맞춰간다.
1번째 학습
예측 → 틀린 정도 계산 → 가중치 조금 수정

2번째 학습
다시 예측 → 다시 틀린 정도 계산 → 가중치 또 수정

3번째 학습
다시 수정

epoch를 반복할수록 같은 데이터를 여러 번 보면서 가중치를 계속 보정한다.

한 번 보고 정답 외우는 게 아니라
문제집을 여러 번 풀면서
틀린 부분을 계속 고치는 과정
(단, 너무 많이 풀면 문제집을 외워 버리므로 EarlyStopping으로 적당한 때 멈춘다)
```

```
처음 파라미터 = 랜덤값
      ↓
문장 64개 입력
      ↓
긍정/부정 예측
      ↓
정답과 비교
      ↓
오차 계산
      ↓
파라미터 수정

이 과정을 계속 반복하면서 랜덤했던 값이 점점 정답을 잘 맞히는 값으로 바뀌는 것이 학습
```


### 🔵 ④ Evaluate → 평가


In [ ]:
# 셀 13. 모델 평가
# 학습에 사용하지 않은 테스트 데이터로 모델 실력 확인
test_loss, test_acc = model.evaluate(X_test_pad, y_test, verbose=0)
print(f"Test Loss:     {test_loss:.4f}")  # 테스트 데이터의 손실값 (얼마나 틀렸는지)
print(f"Test Accuracy: {test_acc:.4f}")   # 테스트 데이터의 정확도 (얼마나 맞혔는지)
print()

# 정확도만 보면 부정 리뷰를 잘 찾는지 알 수 없으므로 클래스별 성능도 확인합니다.
y_pred = (model.predict(X_test_pad, verbose=0)[:, 0] >= 0.5).astype(int)
print(classification_report(y_test, y_pred, target_names=["부정(0)", "긍정(1)"], digits=3))

Test Loss:     0.2140
Test Accuracy: 0.9235

              precision    recall  f1-score   support

       부정(0)      0.854     0.888     0.871      2152
       긍정(1)      0.953     0.938     0.946      5276

    accuracy                          0.924      7428
   macro avg      0.904     0.913     0.908      7428
weighted avg      0.925     0.924     0.924      7428



---
## 🔁 단어 순서를 읽는 모델: Bi-LSTM

위 모델은 단어 벡터의 **평균**을 냅니다. 평균을 내면 단어 순서가 사라지므로,
"não, gostei muito"와 "gostei, não muito"를 같은 문장으로 봅니다. PART 7-1 TF-IDF와 같은 한계입니다.

**LSTM**은 문장을 앞에서부터 한 단어씩 **순서대로** 읽으면서, 앞에서 읽은 내용을 기억합니다.
**Bidirectional(양방향)**은 같은 문장을 뒤에서부터도 한 번 더 읽어 두 결과를 합칩니다.

```
Embedding 평균 :  [produto] [não] [chegou]  →  평균 1개  →  긍정/부정      (순서 정보 없음)
Bi-LSTM       :  produto → não → chegou    →  기억 상태  →  긍정/부정      (순서 반영)
                 produto ← não ← chegou
```

| 설정 | 의미 |
|---|---|
| `mask_zero=True` | 번호 0(패딩)은 **읽지 않고 건너뜁니다.** 문장 뒤에 붙은 0이 기억을 흐리지 않게 합니다. |
| `LSTM(32)` | 읽은 내용을 32개 숫자로 기억 |
| `Bidirectional(...)` | 앞→뒤, 뒤→앞 두 방향으로 읽음 |

> ⚠️ `mask_zero=True`는 LSTM 같은 순서 모델과 함께 씁니다. 위의 평균 모델(GlobalAveragePooling1D)에 붙이면
> 일부 Keras 버전에서 학습이 되지 않는 문제가 있어 평균 모델에는 쓰지 않았습니다.
>
> ⚠️ **GPU 주의**: Colab GPU의 LSTM(cuDNN)은 `mask_zero=True`일 때 **단어가 하나도 없는 문장**(".", "..." 같은 리뷰)을 처리하지 못하고 오류를 냅니다. 그래서 아래 셀에서 그런 문장에 `<OOV>` 하나를 넣어 줍니다. 서빙할 때도 빈 문장이 들어올 수 있으니 같은 처리가 필요합니다.

In [ ]:
# 셀 13-1. Bi-LSTM 모델 정의 · 학습 · 평가 (약 2~4분, GPU 런타임이면 더 빠름)

# ⚠️ 단어가 하나도 없는 리뷰 처리
# "." "..." 처럼 기호만 있는 리뷰는 숫자로 바꾸면 [] → 패딩하면 전부 0이 됩니다.
# mask_zero=True는 0을 건너뛰므로, 이런 문장은 읽을 단어가 하나도 없습니다.
# Colab GPU(cuDNN)의 LSTM은 '전부 건너뛰는 문장'이 있으면 오류를 냅니다.
# → 그런 문장은 맨 앞에 <OOV>(1번) 하나를 넣어 줍니다. (행을 지우지 않으므로 test 데이터는 PART 7-1과 그대로 같습니다)
def fill_empty(X_pad):
    X = X_pad.copy()
    empty = (X == 0).all(axis=1)   # 모든 칸이 0인 문장
    X[empty, 0] = 1                # 첫 칸에 <OOV>(1번) 넣기
    return X, int(empty.sum())

X_train_lstm, n_train = fill_empty(X_train_pad)
X_val_lstm,   n_val   = fill_empty(X_val_pad)
X_test_lstm,  n_test  = fill_empty(X_test_pad)
print(f"단어가 하나도 없는 문장 : train {n_train}개 / validation {n_val}개 / test {n_test}개 → <OOV> 1개로 채움")
print()

tf.keras.utils.set_random_seed(42)

lstm_model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=64, mask_zero=True),  # 0(패딩)은 건너뜀
    Bidirectional(LSTM(32)),       # 앞→뒤, 뒤→앞으로 순서대로 읽기
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
])

lstm_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

lstm_history = lstm_model.fit(
    X_train_lstm, y_train,
    epochs=10, batch_size=64,
    validation_data=(X_val_lstm, y_val),
    class_weight=class_weight,
    callbacks=[EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)],
    verbose=2
)

# 평가 : 앞의 평균 모델과 똑같은 test 데이터 사용
lstm_pred = (lstm_model.predict(X_test_lstm, verbose=0)[:, 0] >= 0.5).astype(int)
print()
print(classification_report(y_test, lstm_pred, target_names=["부정(0)", "긍정(1)"], digits=3))

단어가 하나도 없는 문장 : train 70개 / validation 13개 / test 20개 → <OOV> 1개로 채움

Epoch 1/10
372/372 - 17s - 46ms/step - accuracy: 0.9005 - loss: 0.2806 - val_accuracy: 0.9282 - val_loss: 0.2038
Epoch 2/10
372/372 - 5s - 14ms/step - accuracy: 0.9422 - loss: 0.1675 - val_accuracy: 0.9293 - val_loss: 0.2027
Epoch 3/10
372/372 - 4s - 11ms/step - accuracy: 0.9563 - loss: 0.1361 - val_accuracy: 0.9285 - val_loss: 0.2146
Epoch 4/10
372/372 - 4s - 11ms/step - accuracy: 0.9648 - loss: 0.1151 - val_accuracy: 0.9231 - val_loss: 0.2401

              precision    recall  f1-score   support

       부정(0)      0.819     0.925     0.869      2152
       긍정(1)      0.968     0.917     0.941      5276

    accuracy                          0.919      7428
   macro avg      0.893     0.921     0.905      7428
weighted avg      0.925     0.919     0.920      7428



## 📌 딥러닝 점수, 무엇과 비교해야 할까? (기준선)

정확도 90%가 좋은 점수일까요? **비교 대상**이 있어야 알 수 있습니다.

| 비교 대상 | 설명 |
|---|---|
| 기준선 ① 다수 클래스 | 모든 리뷰를 '긍정'이라고 찍기 (긍정이 71%이므로 정확도 약 71%) |
| 기준선 ② PART 7-1 머신러닝 | TF-IDF + 로지스틱 회귀. PART 7-1에서 저장한 모델을 불러와 **같은 test**로 평가 |
| 딥러닝 ① Embedding 평균 | 셀 10~13 모델 (단어 순서 X) |
| 딥러닝 ② Bi-LSTM | 셀 13-1 모델 (단어 순서 O) |

In [ ]:
# 셀 13-2. 기준선과 비교
import joblib
from sklearn.metrics import accuracy_score, f1_score

# 기준선 ① : 전부 '긍정'이라고 찍기
majority_acc = (y_test == 1).mean()

# 기준선 ② : PART 7-1에서 저장한 머신러닝 모델 (TF-IDF + 로지스틱 회귀)
# PART 7-1과 PART 7-2는 같은 방법(random_state=42)으로 나눴기 때문에 X_test가 똑같습니다.
ml = joblib.load(f"{model_path}/part7_1_sentiment_ml.pkl")
logreg_pred = ml["model"].predict(ml["tfidf"].transform(X_test))

compare = pd.DataFrame({
    "모델": ["기준선① 전부 긍정", "PART 7-1 머신러닝 (TF-IDF)", "딥러닝① Embedding 평균", "딥러닝② Bi-LSTM"],
    "단어 순서": ["-", "X", "X", "O"],
    "정확도": [majority_acc,
              accuracy_score(y_test, logreg_pred),
              accuracy_score(y_test, y_pred),
              accuracy_score(y_test, lstm_pred)],
    "부정 F1": [0.0,
               f1_score(y_test, logreg_pred, pos_label=0),
               f1_score(y_test, y_pred, pos_label=0),
               f1_score(y_test, lstm_pred, pos_label=0)],
})
compare.round(3)

,모델,단어 순서,정확도,부정 F1
0,기준선① 전부 긍정,-,0.710,0.000
1,PART 7-1 머신러닝 (TF-IDF),X,0.917,0.869
2,딥러닝① Embedding 평균,X,0.924,0.871
3,딥러닝② Bi-LSTM,O,0.919,0.869


**결과 해석**
- 두 딥러닝 모델 모두 기준선①(찍기 71%)보다 훨씬 좋습니다. → 리뷰의 단어에서 긍정/부정 패턴을 실제로 배웠습니다.
- 세 모델(PART 7-1 머신러닝, Embedding 평균, Bi-LSTM)의 정확도는 **모두 0.92 안팎으로 거의 같습니다.**
- **Embedding 평균 모델**은 PART 7-1과 정확도는 비슷하지만 부정 F1은 약간 낮습니다. 둘 다 단어 순서를 보지 않기 때문입니다.
- **Bi-LSTM**은 단어 순서를 읽어 부정 F1이 평균 모델보다 조금 높습니다. 하지만 학습 시간은 몇 배 더 걸리고, PART 7-1과 비교하면 **거의 같은 수준**입니다.
- 참고로 PART 7-1 모델은 train 전체로 학습했고, 딥러닝 모델은 train의 80%로 학습했습니다(20%는 validation).
- Olist 리뷰는 대부분 **10단어 안팎으로 짧고**, "bom(좋다)", "não recebi(못 받았다)" 같은 **핵심 단어만으로 판단되는 문제**입니다.
  이런 문제에서는 단어 순서를 읽어도 얻을 정보가 많지 않아, 전통적인 모델도 충분히 강합니다.
  (PART 7-1의 bigram 개선 모델처럼 **입력만 다듬은 머신러닝**이 오히려 가장 높게 나오기도 합니다.)

> **교훈**: 딥러닝이 항상 이기는 것은 아닙니다. 실무에서는 단순한 모델을 먼저 만들어 기준선으로 삼고,
> 딥러닝이 그보다 **확실히** 나을 때 쓰는 것이 원칙입니다. 학습 시간·서버 비용·설명 가능성도 함께 따져야 합니다.
> 딥러닝은 문장이 길고 문맥이 중요할수록(→ PART 7-3 BERT) 강점이 커집니다.


### 🔵 ⑤ Predict → 예측


In [ ]:
# 셀 14. 임의 문장으로 예측 테스트

def predict_sentiment(text):
    """문장이 긍정인지 부정인지 예측합니다."""
    seq    = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")
    prob   = model.predict(padded, verbose=0)[0][0]
    result = "긍정 😊" if prob >= 0.5 else "부정 😞"
    return result, round(float(prob), 3)

print(predict_sentiment("produto muito bom gostei"))  # 좋은 제품이에요
print(predict_sentiment("produto ruim nao gostei"))   # 나쁜 제품이에요
print()

# ⚠️ 한국어 문장은 어떻게 될까?
korean = "배송이 빠르고 상품이 좋아요"
print("한국어 숫자 변환:", tokenizer.texts_to_sequences([korean]))
print(predict_sentiment(korean))
# ▶ 한국어 단어는 단어집에 하나도 없어서 전부 1(<OOV>)로 바뀝니다.
#   모델은 "모르는 단어 4개"만 보고 답을 낸 것이므로, 결과가 긍정이든 부정이든 의미가 없습니다.
#   확률이 높게 나와도 믿으면 안 됩니다. → 학습한 언어(포르투갈어)로만 사용해야 합니다.

('긍정 😊', 0.993)
('부정 😞', 0.083)

한국어 숫자 변환: [[1, 1, 1, 1]]
('긍정 😊', 0.834)


---
## 💾 모델 저장: Tokenizer와 MAX_LEN을 함께 맞춰야 하는 이유

딥러닝 모델의 입력은 **단어 번호 50개짜리 배열**입니다. 새 문장도 학습 때와 **똑같은 번호, 똑같은 길이**로 만들어야 합니다. (셀 14 `predict_sentiment` 함수가 이 순서입니다)

```
새 문장 → tokenizer.texts_to_sequences()  (학습 때 만든 단어 번호)
        → pad_sequences(maxlen=50)          (학습 때와 같은 길이)
        → model.predict()                   → 긍정 확률
```

| 실수 | 결과 |
|---|---|
| Tokenizer를 저장 안 함 | 같은 단어가 다른 번호가 되어 **엉뚱한 예측** |
| MAX_LEN을 다르게 (예: 100) | 입력 모양이 달라져 오류가 나거나 예측이 틀어짐 |
| `padding="post"`를 `"pre"`로 바꿈 | 0이 앞에 붙어 학습 때와 다른 입력이 됨 |

> 📌 머신러닝(PART 7-1)은 한 파일에 함께 저장했지만, Keras 모델은 `.keras` 파일로 따로 저장합니다. 그래서 **모델 파일 + Tokenizer 파일 + MAX_LEN 값** 세 가지를 함께 기억해야 합니다.

### 📦 배포를 생각하면: 파일 크기

| 저장 방법 | 파일 크기 (대략) |
|---|---|
| `model.save()` 그대로 (옵티마이저 포함) | 약 7.5MB |
| 가중치만 옮긴 배포용 모델 (셀 15) | 약 2.5MB |
| 참고: 단어 5,000개 · 임베딩 32차원으로 줄인 모델 | 약 0.7MB |

- Embedding 층의 파라미터는 **단어 수 × 임베딩 차원**(10,000 × 64 = 64만 개)이라 모델 크기 대부분을 차지합니다.
- Olist 리뷰는 짧고 핵심 단어로 판단되기 때문에, 단어 수와 차원을 줄여도 성능이 거의 떨어지지 않습니다. (강사 사전 실험: 정확도 0.909 → 0.917, 실행할 때마다 조금씩 다름) 배포 용량이 중요하면 `MAX_WORDS = 5000`, `output_dim=32`로 바꿔서 다시 학습해 보세요.

In [ ]:
# 셀 15. 모델과 Tokenizer 저장 (배포용)

# 딥러닝 모델 저장 (학습된 가중치와 모델 구조를 저장)
# .keras : 최신 Keras 권장 저장 형식 (.h5는 예전 형식)
# 저장하는 모델은 가볍고 빠른 ① Embedding 평균 모델입니다. (② Bi-LSTM은 비교용으로만 학습했습니다)

# ⚠️ model.save()를 그대로 쓰면 학습 때만 필요한 옵티마이저(adam) 정보까지 저장되어
#    파일이 약 3배 커집니다. 예측에는 가중치만 있으면 되므로,
#    같은 구조의 모델을 새로 만들고 가중치만 옮겨서 저장합니다. (예측 결과는 똑같습니다)
from tensorflow.keras.layers import Input

serve_model = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(input_dim=MAX_WORDS, output_dim=64),
    GlobalAveragePooling1D(),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
])
serve_model.set_weights(model.get_weights())   # 학습한 가중치 복사

# 두 모델의 예측이 같은지 확인
same = np.allclose(model.predict(X_test_pad[:100], verbose=0), serve_model.predict(X_test_pad[:100], verbose=0))
print("학습 모델과 배포용 모델의 예측이 같은가?", same)

serve_model.save(f"{model_path}/part7_2_sentiment_model.keras")

# Tokenizer 저장 (단어집 정보가 담겨있음)
# 문장을 숫자로 바꾸는 단어집(Tokenizer) 저장
# 어떤 단어가 몇 번인지 기억한 정보 저장
with open(f"{model_path}/part7_2_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("감성분석 모델 저장 완료!")
print("저장 위치:", model_path)
for name in ["part7_2_sentiment_model.keras", "part7_2_tokenizer.pkl"]:
    print(f"  - {name:32s} {os.path.getsize(f'{model_path}/{name}') / 1024:,.0f} KB")
print(f"  (불러올 때도 같은 MAX_LEN = {MAX_LEN} 으로 패딩해야 합니다)")


학습 모델과 배포용 모델의 예측이 같은가? True
감성분석 모델 저장 완료!
저장 위치: /content/drive/MyDrive/Colab Notebooks/Data_Analysis/Olist/model
  - part7_2_sentiment_model.keras    2,541 KB
  - part7_2_tokenizer.pkl            463 KB
  (불러올 때도 같은 MAX_LEN = 50 으로 패딩해야 합니다)


---

## 마무리 정리

| 확인할 내용 | 설명 |
|---|---|
| 이 파트의 핵심 | 문장 → 숫자 → 딥러닝으로 긍정/부정을 예측합니다. |
| test 원칙 | test 데이터는 절대 수정하지 않습니다. (시험지 역할) |
| validation 원칙 | validation도 학습 전에 따로 떼어 두고 수정하지 않습니다. 데이터를 복제(증폭)할 때는 반드시 떼어 둔 **뒤에** 해야 합니다. |
| 불균형 보정 | class_weight로 적은 클래스(부정)를 더 중요하게 학습합니다. |
| MAX_LEN | 문장 길이 분포를 보고 정합니다. 너무 길면 0(패딩)만 늘어납니다. |
| EarlyStopping | 검증 성능이 멈추면 학습을 멈추고 가장 좋았던 시점으로 되돌립니다. 마지막 epoch 운에 결과가 좌우되지 않습니다. |
| 순서 모델 | Embedding 평균은 단어 순서를 보지 않습니다. 순서를 보려면 LSTM 같은 층이 필요합니다. |
| 기준선 비교 | 딥러닝 점수도 단순 모델(TF-IDF + 로지스틱 회귀)과 비교해야 의미를 알 수 있습니다. |
| 언어 한계 | 학습한 언어(포르투갈어)가 아닌 문장은 전부 <OOV>가 되어 예측이 의미 없습니다. |
| 모델 저장 | 모델 파일(.keras) + Tokenizer + MAX_LEN을 학습 때와 똑같이 맞춰야 새 문장을 예측할 수 있습니다. |
| 모델 저장 | 배포용은 옵티마이저 정보를 빼고 가중치만 저장합니다. (약 7.5MB → 2.5MB) |
| 다음 단계 | PART 7-3에서 사전학습된 BERT 모델로 같은 리뷰를 분류하고 PART 7-1·7-2와 비교합니다. |

```
define/model 구성 → 문제집/교재 구성
compile          → 채점 기준 + 공부 방법 정하기
fit              → 실제로 문제를 풀면서 학습
evaluate         → 시험 보고 성적 확인

model = Sequential(...)   # 어떤 문제집으로 공부할지 구성
model.compile(...)        # 채점 기준과 학습 방법 정하기
model.fit(...)            # 실제 학습


| 구분                 | 사용하는 데이터             | 역할                | 쉽게 비유   |
| ------------------ | -------------------- | ----------------- | ------- |
| **Train 데이터**      | `X_train`, `y_train` | 모델이 실제로 학습하는 데이터 (class_weight로 불균형 보정) | 공부할 문제집 |
| **Validation 데이터** | `X_val`, `y_val`     | 학습 중간에 성능 확인 | 중간시험    |
| **Test 데이터**       | `X_test`, `y_test`   | 학습이 끝난 후 최종 성능 확인 | 최종시험    |


```
흐름

전체 데이터
   ↓
train_test_split()  ① 시험지(test) 먼저 분리
   ↓
Train 데이터                         Test 데이터
   ↓                                    ↓
train_test_split()  ② 중간시험(validation) 분리
   ↓
Train (80%)        Validation (20%)
   ↓                    ↓
③ class_weight 적용     그대로 둠
   ↓                    ↓
fit()  ← 학습 + 매 epoch 검증            evaluate() ← 최종시험


model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val)
)

처음부터 따로 빼놓은 Test 데이터로 최종 평가
model.evaluate(X_test_pad, y_test)

Train = 공부, Validation = 중간시험, Test = 최종시험
시험지(Test)와 중간시험(Validation)은 절대 수정하지 않는다
```